# Model initiation

In [1]:
from extraction_tools import *
from models import *
from glob import glob

save_dir = r'D:\AIC_NEW_EXTRACTION'
keyframe_extraction_model = TransNetV2Model(device="cuda")
keyframe_feature_extraction_model = SigLIP2Model(name="google/siglip2-so400m-patch14-384", device="cuda")
transcription_extraction_model = FasterWhisperModel(name="turbo", device="cuda")
transcription_feature_extraction_model = BGEM3Model(name="BAAI/bge-m3", device="cuda")

# keyframe_extractor = KeyframeAndFeatureExtractor(
#     keyframe_extraction_model,
#     keyframe_feature_extraction_model,
#     save_dir
# )
transcription_extractor = TranscriptFeatureProcessor(
    transcription_extraction_model,
    transcription_feature_extraction_model,
    save_dir
)


[INFO] Loading quantized turbo
[INFO] turbo is on cuda device
[INFO] Loading quantized BAAI/bge-m3
[INFO] BAAI/bge-m3 is on cuda device


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

# Extract keyframes

In [ ]:
vid_paths = glob(fr"")

keyframe_extractor.extract_and_save_keyframes(
    vid_paths,
    feature_extraction_batch_size=32,
    shot_confidence_threshold=0.7,
    clustering_similarity_threshold=0.9,
    max_workers=4
)

# Extract transcripts

In [6]:
vid_paths = glob("E:/AIC_VID_ONLY/vids/*/L3*.mp4")
map_paths = glob("D:/AIC_NEW_EXTRACTION/*/L3*.csv")

if len(vid_paths) != len(map_paths): raise RuntimeError(f"Mismatch videos and maps count: {len(vid_paths)} to {len(map_paths)}")

transcription_extractor.extract_and_save_transcript(
    vid_paths,
    map_paths,
    transcription_batch_size=16,
    transcription_language="vi",
    alignment_max_time_distance=10,
    feature_extraction_batch_size=32,
    max_workers=4
)

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 730.40it/s]1:59<01:36,  1.75s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<?, ?it/s] 44/96 [02:09<01:57,  2.25s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 281.69it/s]4:29<00:28,  3.20s/it]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 2000.14it/s]

Inference Embeddings: 100%|██████████| 2/2 [00:00<00:00,  4.90it/s]

Extracting transcripts from video: 100%|██████████| 96/96 [04:53<00:00,  3.05s/it]


In [5]:
feats = [np.load(path) for path in glob(r"D:\AIC_NEW_EXTRACTION\feats\bgem3_dense\L21_V001\*.npy")]
for f in feats: print(f, len(f))

[-0.01916504 -0.01300049 -0.00173187 ... -0.02062988 -0.02331543
 -0.00227356] 1024
[-0.01696777 -0.00628662 -0.0534668  ... -0.01141357  0.0111084
 -0.01989746] 1024
[-0.01696777 -0.00628662 -0.0534668  ... -0.01141357  0.0111084
 -0.01989746] 1024
[-0.01696777 -0.00628662 -0.0534668  ... -0.01141357  0.0111084
 -0.01989746] 1024
[-0.00393677  0.05102539 -0.0703125  ...  0.03125     0.03808594
 -0.0145874 ] 1024
[-0.00192261  0.04711914 -0.06689453 ...  0.03051758  0.03759766
 -0.01379395] 1024
[-0.00506592  0.06347656 -0.0559082  ...  0.02685547  0.02319336
 -0.03271484] 1024
[-0.02978516 -0.00646973 -0.03686523 ... -0.0065918  -0.02624512
  0.00215149] 1024
[-0.01196289  0.04370117 -0.01391602 ...  0.02246094  0.01928711
 -0.01293945] 1024
[-0.00750732  0.03930664 -0.01416016 ...  0.01831055  0.02197266
  0.00228882] 1024
[ 0.01531982  0.07275391 -0.01464844 ... -0.01843262  0.01843262
 -0.00091553] 1024
[ 0.00653076  0.04321289 -0.03149414 ... -0.01831055  0.01745605
  0.02197266] 

# Test

In [1]:
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from extraction_tools import BEiT3Model

img_paths = [str(path) for path in list(Path(r"D:\AIC_NEW_EXTRACTION").glob("imgs/**/*.jpg"))]

beit3_model = BEiT3Model("timm/beit3_large_patch16_224.pt", device="cuda")
beit3_model.load()

[INFO] Loading quantized timm/beit3_large_patch16_224.pt
[INFO] timm/beit3_large_patch16_224.pt is on cuda device


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] Requested torchvision backend is not available. Falling back to pil backend.


In [2]:
batch_size = 1024
feats = []
for i in tqdm(list(range(0, len(img_paths), batch_size))):
    j = i + batch_size if i + batch_size < len(img_paths) else len(img_paths)
    imgs = [Image.open(path) for path in img_paths[i:j]]
    feat = beit3_model.run(imgs, batch_size=batch_size)
    for f in feat: feats.append(f)

100%|██████████| 149/149 [1:08:22<00:00, 27.53s/it]


In [4]:
import os
import numpy as np
feat_dir = Path(r"D:/AIC_NEW_EXTRACTION/feats/beit3_dense/")
for path, feat in zip(img_paths, feats):
    path = Path(path)
    if not os.path.exists(os.path.join(feat_dir, path.parts[-2])):
        os.mkdir(os.path.join(feat_dir, path.parts[-2]))
    np.save(str(Path(feat_dir) / path.parts[-2] / path.parts[-1]), feat)

In [ ]:
from glob import glob
import os
beit3_dense_paths = glob(r"D:\AIC26\feats\beit3_dense\*\*.npy")
for path in beit3_dense_paths:
    new_path = path.replace(".jpg", "")
    os.rename(path, new_path)